# Day Feature Engineering: 
## `mart_report_daily_adoption`

This notebook builds the first draft of `mart_report_daily_adoption` from `data/processed/fact_report_views.csv`.

Day objective:
- Aggregate report views to the daily `date` and `report_id` grain
- Create `daily_views`, `unique_viewers`, and `views_per_user`
- Build a per-report daily date spine from each report's launch date to the dataset's global max date
- Fill missing post-launch days with zeros and save the result for later feature engineering and forecasting work


## 1. Setup

We define project paths and import the reusable feature engineering helpers.


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.engagement_features import build_user_engagement_features
from src.features.report_features import (
    add_time_series_usage_features,
    build_report_daily_adoption,
)

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FACT_REPORT_VIEWS_PATH = DATA_PROCESSED_DIR / "fact_report_views.csv"
FACT_PAGE_VIEWS_PATH = DATA_PROCESSED_DIR / "fact_page_views.csv"
OUTPUT_PATH = DATA_PROCESSED_DIR / "mart_report_daily_adoption.csv"
TS_OUTPUT_PATH = DATA_PROCESSED_DIR / "mart_report_daily_adoption_ts_features.csv"
ENGAGEMENT_OUTPUT_PATH = DATA_PROCESSED_DIR / "mart_user_engagement.csv"


## 2. Load `fact_report_views`

The processed report view fact table already created in the repo.


In [2]:
if not FACT_REPORT_VIEWS_PATH.exists():
    raise FileNotFoundError(f"Expected input file was not found: {FACT_REPORT_VIEWS_PATH}")

fact_report_views = pd.read_csv(FACT_REPORT_VIEWS_PATH)
print(f"Loaded input from: {FACT_REPORT_VIEWS_PATH}")
print("Shape:", fact_report_views.shape)
display(fact_report_views.head())


Loaded input from: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/fact_report_views.csv
Shape: (411006, 6)


,date_key,report_id,user_key,consumption_method,distribution_method,view_count
0,20250101,R_001,UK_0002,Web,Direct,1
1,20250101,R_001,UK_0004,Web,App,3
2,20250101,R_001,UK_0012,Mobile,App,2
3,20250101,R_001,UK_0019,Web,Direct,1
4,20250101,R_001,UK_0023,Web,Direct,1


## 3. Aggregate to the Daily Report Grain

We first aggregate the event-level fact table to `date` and `report_id`, using `view_count` for total daily views and `user_key` for distinct viewers.


In [3]:
report_daily_base = build_report_daily_adoption(
    fact_report_views=fact_report_views,
    date_col="date_key",
    report_col="report_id",
    user_col="user_key",
    views_col="view_count",
)

display(report_daily_base.head())
print("Shape:", report_daily_base.shape)


,date,report_id,daily_views,unique_viewers,views_per_user
0,2025-01-01,R_001,36,22,1.636364
1,2025-01-02,R_001,41,25,1.640000
2,2025-01-03,R_001,37,27,1.370370
3,2025-01-04,R_001,17,12,1.416667
4,2025-01-05,R_001,30,18,1.666667


Shape: (13627, 5)


## 4. Build the Per-Report Daily Date Spine

Each report starts on its first observed view date. Every report then continues daily until the dataset's global maximum date. Missing post-launch dates are filled with zeros.


In [4]:
report_start_dates = (
    report_daily_base.groupby("report_id", as_index=False)["date"]
    .min()
    .rename(columns={"date": "start_date"})
)
global_max_date = report_daily_base["date"].max()

date_spine_frames = []
for row in report_start_dates.itertuples(index=False):
    report_dates = pd.date_range(start=row.start_date, end=global_max_date, freq="D")
    date_spine_frames.append(
        pd.DataFrame({"date": report_dates, "report_id": row.report_id})
    )

report_date_spine = pd.concat(date_spine_frames, ignore_index=True)

mart_report_daily_adoption = report_date_spine.merge(
    report_daily_base,
    on=["date", "report_id"],
    how="left",
)

mart_report_daily_adoption[["daily_views", "unique_viewers"]] = (
    mart_report_daily_adoption[["daily_views", "unique_viewers"]]
    .fillna(0)
    .astype(int)
)

mart_report_daily_adoption["views_per_user"] = (
    mart_report_daily_adoption["daily_views"]
    .div(mart_report_daily_adoption["unique_viewers"].replace(0, pd.NA))
    .fillna(0.0)
)

mart_report_daily_adoption = mart_report_daily_adoption[
    ["date", "report_id", "daily_views", "unique_viewers", "views_per_user"]
].sort_values(["report_id", "date"]).reset_index(drop=True)

display(mart_report_daily_adoption.head())
print("Shape:", mart_report_daily_adoption.shape)
print("Global max date:", global_max_date.date())


/var/folders/_r/0mbwmsn56rl_rr8h_2gs0tww0000gn/T/ipykernel_46877/2147230076.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0.0)


,date,report_id,daily_views,unique_viewers,views_per_user
0,2025-01-01,R_001,36,22,1.636364
1,2025-01-02,R_001,41,25,1.640000
2,2025-01-03,R_001,37,27,1.370370
3,2025-01-04,R_001,17,12,1.416667
4,2025-01-05,R_001,30,18,1.666667


Shape: (13650, 5)
Global max date: 2026-03-31


## 5. Validation Checks

These checks confirm the final mart has the expected grain, no pre-launch rows, no nulls in key columns, and a consistent end date across reports.


In [5]:
grain_is_unique = not mart_report_daily_adoption.duplicated(subset=["date", "report_id"]).any()
null_counts = mart_report_daily_adoption[
    ["date", "report_id", "daily_views", "unique_viewers", "views_per_user"]
].isnull().sum()

validation_dates = mart_report_daily_adoption.merge(report_start_dates, on="report_id", how="left")
has_prelaunch_rows = (validation_dates["date"] < validation_dates["start_date"]).any()
report_end_dates = mart_report_daily_adoption.groupby("report_id")["date"].max()
all_reports_end_at_global_max = report_end_dates.eq(global_max_date).all()
zero_view_logic_holds = (
    mart_report_daily_adoption.loc[mart_report_daily_adoption["unique_viewers"] == 0, "views_per_user"]
    .eq(0)
    .all()
)

print("One row per date/report_id:", grain_is_unique)
print("No rows before a report's first observed view date:", not has_prelaunch_rows)
print("All reports end at the dataset global max date:", all_reports_end_at_global_max)
print("views_per_user is 0 when unique_viewers is 0:", zero_view_logic_holds)

print("\nNull counts in key columns:")
display(null_counts.to_frame(name="null_count"))


One row per date/report_id: True
No rows before a report's first observed view date: True
All reports end at the dataset global max date: True
views_per_user is 0 when unique_viewers is 0: True

Null counts in key columns:


,null_count
date,0
report_id,0
daily_views,0
unique_viewers,0
views_per_user,0


## 6. Summary Statistics

A quick distribution check helps confirm the current features look reasonable before saving the mart.


In [6]:
summary_stats = mart_report_daily_adoption[
    ["daily_views", "unique_viewers", "views_per_user"]
].describe()
display(summary_stats)


,daily_views,unique_viewers,views_per_user
count,13650.000000,13650.000000,13650.000000
mean,46.682857,30.110330,1.547944
std,31.419843,20.007464,0.230424
min,0.000000,0.000000,0.000000
25%,24.000000,16.000000,1.424242
50%,41.000000,26.000000,1.538462
75%,62.000000,39.750000,1.666667
max,201.000000,119.000000,4.000000


## Time-Series Usage Features

Rolling metrics help us capture short-term and medium-term usage momentum for each report without changing the underlying daily grain. They are especially useful for spotting recent growth, softening demand, and stable recurring usage patterns.

A complete daily calendar matters because rolling windows should include true zero-usage days rather than skipping missing dates. That keeps the 7-day and 28-day features consistent across reports and makes later forecasting features more trustworthy.

Week-over-week change compares a report's current `daily_views` to its value 7 days earlier using `(current - lag_7d) / lag_7d`. When the 7-day lag is missing or zero, the result stays `NaN` so we avoid invalid infinite values.


In [7]:
mart_report_daily_adoption_ts = add_time_series_usage_features(mart_report_daily_adoption)

display(mart_report_daily_adoption_ts.head())
print("Shape:", mart_report_daily_adoption_ts.shape)


,date,report_id,daily_views,unique_viewers,views_per_user,views_7d,views_28d,viewers_7d,viewers_28d,wow_change_views
0,2025-01-01,R_001,36,22,1.636364,36.0,36.0,22.0,22.0,NaN
1,2025-01-02,R_001,41,25,1.640000,77.0,77.0,47.0,47.0,NaN
2,2025-01-03,R_001,37,27,1.370370,114.0,114.0,74.0,74.0,NaN
3,2025-01-04,R_001,17,12,1.416667,131.0,131.0,86.0,86.0,NaN
4,2025-01-05,R_001,30,18,1.666667,161.0,161.0,104.0,104.0,NaN


Shape: (13650, 10)


### Sample Report Time Series

Looking at a small sample of reports makes it easier to sanity-check the rolling features and week-over-week calculations.


In [8]:
sample_reports = mart_report_daily_adoption_ts["report_id"].drop_duplicates().head(2).tolist()
sample_time_series = mart_report_daily_adoption_ts.loc[
    mart_report_daily_adoption_ts["report_id"].isin(sample_reports),
    [
        "date",
        "report_id",
        "daily_views",
        "views_7d",
        "views_28d",
        "wow_change_views",
        "viewers_7d",
        "viewers_28d",
    ],
]
display(sample_time_series.head(20))


,date,report_id,daily_views,views_7d,views_28d,wow_change_views,viewers_7d,viewers_28d
0,2025-01-01,R_001,36,36.0,36.0,NaN,22.0,22.0
1,2025-01-02,R_001,41,77.0,77.0,NaN,47.0,47.0
2,2025-01-03,R_001,37,114.0,114.0,NaN,74.0,74.0
3,2025-01-04,R_001,17,131.0,131.0,NaN,86.0,86.0
4,2025-01-05,R_001,30,161.0,161.0,NaN,104.0,104.0
5,2025-01-06,R_001,42,203.0,203.0,NaN,129.0,129.0
6,2025-01-07,R_001,41,244.0,244.0,NaN,153.0,153.0
7,2025-01-08,R_001,36,244.0,280.0,0.000000,160.0,182.0
8,2025-01-09,R_001,43,246.0,323.0,0.048780,164.0,211.0
9,2025-01-10,R_001,43,252.0,366.0,0.162162,166.0,240.0


### Time-Series Validation

These checks confirm that the time-series step enriches the existing dataset without changing row count or introducing invalid values.


In [9]:
row_count_unchanged = len(mart_report_daily_adoption_ts) == len(mart_report_daily_adoption)
ts_grain_is_unique = not mart_report_daily_adoption_ts.duplicated(subset=["date", "report_id"]).any()
rolling_null_counts = mart_report_daily_adoption_ts[["views_7d", "views_28d", "viewers_7d", "viewers_28d"]].isnull().sum()
rolling_columns_populated = rolling_null_counts.eq(0).all()
wow_has_infinite_values = mart_report_daily_adoption_ts["wow_change_views"].isin([float("inf"), float("-inf")]).any()
ts_summary_stats = mart_report_daily_adoption_ts[["views_7d", "views_28d", "wow_change_views"]].describe()
wow_nan_examples = mart_report_daily_adoption_ts.loc[
    mart_report_daily_adoption_ts["wow_change_views"].isna(),
    ["date", "report_id", "daily_views", "views_7d", "views_28d", "wow_change_views"],
].head(10)

print("Row count unchanged:", row_count_unchanged)
print("One row per date/report_id:", ts_grain_is_unique)
print("Rolling columns populated:", rolling_columns_populated)
print("\nNull counts for rolling columns:")
display(rolling_null_counts.to_frame(name="null_count"))
print("wow_change_views contains no infinite values:", not wow_has_infinite_values)

print("\nSummary statistics:")
display(ts_summary_stats)

print("Example rows where wow_change_views is NaN:")
display(wow_nan_examples)


Row count unchanged: True
One row per date/report_id: True
Rolling columns populated: True

Null counts for rolling columns:


,null_count
views_7d,0
views_28d,0
viewers_7d,0
viewers_28d,0


wow_change_views contains no infinite values: True

Summary statistics:


,views_7d,views_28d,wow_change_views
count,13650.000000,13650.000000,13418.000000
mean,324.618315,1268.224908,0.083449
std,197.150328,788.564273,0.589050
min,5.000000,5.000000,-1.000000
25%,189.000000,731.000000,-0.181818
50%,294.000000,1153.000000,0.000000
75%,392.000000,1528.000000,0.222222
max,1052.000000,4013.000000,29.000000


Example rows where wow_change_views is NaN:


,date,report_id,daily_views,views_7d,views_28d,wow_change_views
0,2025-01-01,R_001,36,36.0,36.0,NaN
1,2025-01-02,R_001,41,77.0,77.0,NaN
2,2025-01-03,R_001,37,114.0,114.0,NaN
3,2025-01-04,R_001,17,131.0,131.0,NaN
4,2025-01-05,R_001,30,161.0,161.0,NaN
5,2025-01-06,R_001,42,203.0,203.0,NaN
6,2025-01-07,R_001,41,244.0,244.0,NaN
455,2025-01-01,R_002,80,80.0,80.0,NaN
456,2025-01-02,R_002,54,134.0,134.0,NaN
457,2025-01-03,R_002,58,192.0,192.0,NaN


## 7. Save the Enriched Output

The base Day 1 mart remains available in-memory as `mart_report_daily_adoption`, and the time-series enriched version is saved separately with a clear name.


In [10]:
mart_report_daily_adoption_ts.to_csv(TS_OUTPUT_PATH, index=False)
print(f"Saved mart_report_daily_adoption_ts to: {TS_OUTPUT_PATH}")


Saved mart_report_daily_adoption_ts to: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/mart_report_daily_adoption_ts_features.csv


## Behavioural Features

Usage features tell us how much activity a report receives. Behavioural features tell us how people use that report, including whether users return, whether a small group dominates usage, how long a report sits idle between active days, and how deep people navigate into pages.

These metrics matter because similar view totals can mask very different engagement patterns. A high repeat user rate suggests recurring habits, a high concentration share suggests dependence on a small group of power users, and page-depth helps separate quick glances from deeper exploration.

This section builds:
- `repeat_user_rate`
- `top_10pct_user_share`
- `days_since_last_use`
- `avg_pages_per_user`


### Load Behavioural Inputs

We reuse `fact_report_views` and load `fact_page_views` from the processed layer. In this repo, `date_key` and `user_key` play the role of the generic `date` and `user_id` columns.


In [11]:
if not FACT_PAGE_VIEWS_PATH.exists():
    raise FileNotFoundError(f"Expected input file was not found: {FACT_PAGE_VIEWS_PATH}")

fact_page_views = pd.read_csv(FACT_PAGE_VIEWS_PATH)
print(f"Loaded page views from: {FACT_PAGE_VIEWS_PATH}")
print("Shape:", fact_page_views.shape)
display(fact_page_views.head())


Loaded page views from: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/fact_page_views.csv
Shape: (879149, 7)


,date_key,report_id,page_key,user_key,client,session_source,page_view_count
0,20250101,R_001,6,UK_0002,Browser,Direct,1
1,20250101,R_001,8,UK_0002,Browser,Direct,1
2,20250101,R_001,7,UK_0004,Browser,App,1
3,20250101,R_001,8,UK_0012,Browser,App,1
4,20250101,R_001,3,UK_0012,Browser,Direct,1


### Build `mart_user_engagement`

The engagement mart stays separate from `mart_report_daily_adoption` and focuses only on behavioural signals. For the page-depth proxy, the processed page-view fact uses `page_key` instead of `section_id`, which is sufficient for counting per-user page activity.


In [13]:
mart_user_engagement = build_user_engagement_features(
    fact_report_views=fact_report_views,
    fact_page_views=fact_page_views,
    date_col="date_key",
    report_col="report_id",
    user_col="user_key",
)

display(mart_user_engagement.head())
print("Shape:", mart_user_engagement.shape)


,date,report_id,repeat_user_rate,top_10pct_user_share,days_since_last_use,avg_pages_per_user
0,2025-01-01,R_001,0.000000,0.277778,0,2.181818
1,2025-01-02,R_001,0.120000,0.219512,1,2.120000
2,2025-01-03,R_001,0.407407,0.243243,1,2.370370
3,2025-01-04,R_001,0.583333,0.294118,1,1.916667
4,2025-01-05,R_001,0.500000,0.266667,1,2.277778


Shape: (13627, 6)


### Behavioural Validation

These checks confirm the mart stays at the expected grain and that each behavioural feature remains within a sensible range.


In [14]:
engagement_grain_is_unique = not mart_user_engagement.duplicated(subset=["date", "report_id"]).any()
engagement_null_counts = mart_user_engagement[
    [
        "date",
        "report_id",
        "repeat_user_rate",
        "top_10pct_user_share",
        "days_since_last_use",
        "avg_pages_per_user",
    ]
].isnull().sum()
repeat_rate_in_range = mart_user_engagement["repeat_user_rate"].between(0, 1).all()
top_share_in_range = mart_user_engagement["top_10pct_user_share"].between(0, 1).all()
days_since_non_negative = mart_user_engagement["days_since_last_use"].ge(0).all()
avg_pages_non_negative = mart_user_engagement["avg_pages_per_user"].ge(0).all()

print("One row per date/report_id:", engagement_grain_is_unique)
print("No duplicate keys:", engagement_grain_is_unique)
print("No nulls in key columns:", engagement_null_counts[["date", "report_id"]].eq(0).all())
print("repeat_user_rate between 0 and 1:", repeat_rate_in_range)
print("top_10pct_user_share between 0 and 1:", top_share_in_range)
print("days_since_last_use >= 0:", days_since_non_negative)
print("avg_pages_per_user >= 0:", avg_pages_non_negative)

print("\nNull counts:")
display(engagement_null_counts.to_frame(name="null_count"))


One row per date/report_id: True
No duplicate keys: True
No nulls in key columns: True
repeat_user_rate between 0 and 1: True
top_10pct_user_share between 0 and 1: True
days_since_last_use >= 0: True
avg_pages_per_user >= 0: True

Null counts:


,null_count
date,0
report_id,0
repeat_user_rate,0
top_10pct_user_share,0
days_since_last_use,0
avg_pages_per_user,0


### Behavioural Examples

We inspect a few reports over time and highlight example rows where repeat usage changes, concentration is high, and the gap since last use becomes larger.


In [15]:
example_reports = mart_user_engagement["report_id"].drop_duplicates().head(3).tolist()
engagement_examples = mart_user_engagement.loc[
    mart_user_engagement["report_id"].isin(example_reports),
    [
        "date",
        "report_id",
        "repeat_user_rate",
        "top_10pct_user_share",
        "days_since_last_use",
        "avg_pages_per_user",
    ],
]
display(engagement_examples.head(30))

repeat_rate_changes = mart_user_engagement.groupby("report_id")["repeat_user_rate"].diff().fillna(0).ne(0)
changing_repeat_examples = mart_user_engagement.loc[
    repeat_rate_changes,
    ["date", "report_id", "repeat_user_rate"],
].head(10)
high_concentration_examples = mart_user_engagement.loc[
    mart_user_engagement["top_10pct_user_share"] >= 0.5,
    ["date", "report_id", "top_10pct_user_share", "repeat_user_rate"],
].head(10)
days_since_spike_examples = mart_user_engagement.loc[
    mart_user_engagement["days_since_last_use"] >= 2,
    ["date", "report_id", "days_since_last_use", "repeat_user_rate"],
].head(10)

print("Example rows where repeat_user_rate changes:")
display(changing_repeat_examples)

print("Example rows with high concentration:")
display(high_concentration_examples)

print("Example rows where days_since_last_use spikes:")
display(days_since_spike_examples)


,date,report_id,repeat_user_rate,top_10pct_user_share,days_since_last_use,avg_pages_per_user
0,2025-01-01,R_001,0.000000,0.277778,0,2.181818
1,2025-01-02,R_001,0.120000,0.219512,1,2.120000
2,2025-01-03,R_001,0.407407,0.243243,1,2.370370
3,2025-01-04,R_001,0.583333,0.294118,1,1.916667
4,2025-01-05,R_001,0.500000,0.266667,1,2.277778
5,2025-01-06,R_001,0.360000,0.238095,1,2.600000
6,2025-01-07,R_001,0.541667,0.243902,1,1.791667
7,2025-01-08,R_001,0.689655,0.222222,1,2.103448
8,2025-01-09,R_001,0.758621,0.209302,1,1.931034
9,2025-01-10,R_001,0.793103,0.255814,1,2.344828


Example rows where repeat_user_rate changes:


,date,report_id,repeat_user_rate
1,2025-01-02,R_001,0.120000
2,2025-01-03,R_001,0.407407
3,2025-01-04,R_001,0.583333
4,2025-01-05,R_001,0.500000
5,2025-01-06,R_001,0.360000
6,2025-01-07,R_001,0.541667
7,2025-01-08,R_001,0.689655
8,2025-01-09,R_001,0.758621
9,2025-01-10,R_001,0.793103
10,2025-01-11,R_001,0.923077


Example rows with high concentration:


,date,report_id,top_10pct_user_share,repeat_user_rate
955,2025-02-15,R_003,0.500000,1.00
1005,2025-04-06,R_003,0.666667,1.00
1123,2025-08-02,R_003,0.666667,1.00
1194,2025-10-12,R_003,0.500000,1.00
3791,2025-06-01,R_009,0.500000,1.00
4070,2026-03-07,R_009,1.000000,1.00
5471,2025-01-12,R_013,0.500000,0.25
5477,2025-01-18,R_013,0.750000,1.00
5478,2025-01-19,R_013,0.500000,1.00
5481,2025-01-22,R_013,0.666667,0.50


Example rows where days_since_last_use spikes:


,date,report_id,days_since_last_use,repeat_user_rate
5723,2025-09-22,R_013,2,1.00
8195,2025-01-08,R_019,2,0.20
8207,2025-01-21,R_019,2,0.00
8239,2025-02-24,R_019,3,0.75
8257,2025-03-15,R_019,2,1.00
8299,2025-04-27,R_019,2,1.00
8334,2025-06-02,R_019,2,1.00
8374,2025-07-13,R_019,2,1.00
8415,2025-08-24,R_019,2,1.00
8421,2025-08-31,R_019,2,1.00


### Save `mart_user_engagement`

The behavioural mart is saved separately so it can evolve independently from the adoption and time-series outputs.


In [16]:
mart_user_engagement.to_csv(ENGAGEMENT_OUTPUT_PATH, index=False)
print(f"Saved mart_user_engagement to: {ENGAGEMENT_OUTPUT_PATH}")


Saved mart_user_engagement to: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/mart_user_engagement.csv
